In [ ]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 113.7 MB/s eta 0:00:00


####generation of router_labels

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import faiss
import pickle
import pandas as pd
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

print("1. Loading Models for the Ablation Study...")
# Make sure your LLM is loaded here!
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

embedder = SentenceTransformer('all-MiniLM-L6-v2')

print("2. Loading the 1,000 Labeled Questions...")
dataset = load_dataset("pubmed_qa", "pqa_labeled", split="train")
df = pd.DataFrame(dataset)

# Arrays to hold our data
historical_vectors = []
historical_labels = []

print("3. Generating the History Book (This takes ~10 mins on GPU)...")
for idx, row in tqdm(df.iterrows(), total=len(df)):
    question = row['question']
    true_decision = row['final_decision'].strip().lower()

    # --- A. Semantic Embedding ---
    semantic_vector = embedder.encode([question])[0]

    # --- B. Uncertainty Extraction ---
    context_str = " ".join(row['context']['contexts'])
    prompt = f"Context: {context_str}\nQuestion: {question}\nAnswer (Yes/No/Maybe):"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=10,
            return_dict_in_generate=True, output_scores=True,
            pad_token_id=tokenizer.eos_token_id
        )

    pred_text = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True).lower()

    token_entropies = []
    for step_logits in outputs.scores:
        probs = F.softmax(step_logits[0], dim=-1)
        entropy = -torch.sum(probs * torch.log(probs + 1e-10))
        token_entropies.append(entropy.item())

    mean_ent = np.mean(token_entropies)
    var_ent = np.var(token_entropies)

    # --- C. Fuse Features ---
    state_vector = np.concatenate((semantic_vector, [mean_ent, var_ent]))
    historical_vectors.append(state_vector)

    # --- D. Assign Ground Truth Label ---
    if true_decision in pred_text:
        historical_labels.append(0) # 0 = Factual / Safe
    else:
        historical_labels.append(1) # 1 = Hallucination / Trigger RAG

print("\n4. Building the FAISS Router Index...")
# Convert to float32 matrix for FAISS
vector_matrix = np.array(historical_vectors).astype('float32')
label_array = np.array(historical_labels)

# Build the Index
router_index = faiss.IndexFlatL2(386)
router_index.add(vector_matrix)

print("5. Saving Files to Disk...")
faiss.write_index(router_index, "router_history.index")

with open("router_labels.pkl", "wb") as f:
    pickle.dump(label_array, f)

print("✅ DONE! 'router_history.index' and 'router_labels.pkl' ready to use!")

In [ ]:
import faiss
import numpy as np

print("1. Loading the overpowered text index...")
old_index = faiss.read_index("router_history.index")
num_vectors = old_index.ntotal

print("2. Extracting vectors to fix Feature Scaling...")
# Rip all 1,000 vectors out of the database
all_vectors = np.array([old_index.reconstruct(i) for i in range(num_vectors)])

# Columns 0 to 383 are text.
# Column 384 is Mean Entropy. Column 385 is Varentropy.
# We multiply the entropy by 50 so it commands respect in the vector space!
weight_factor = 50.0
all_vectors[:, 384] = all_vectors[:, 384] * weight_factor
all_vectors[:, 385] = all_vectors[:, 385] * weight_factor

print("3. Building the fixed, balanced index...")
new_index = faiss.IndexFlatL2(386)
new_index.add(all_vectors)

# Save the new weighted index
faiss.write_index(new_index, "router_history_weighted.index")
print("✅ Fixed! Saved as 'router_history_weighted.index'")

1. Loading the overpowered text index...
2. Extracting vectors to fix Feature Scaling...
3. Building the fixed, balanced index...
✅ Fixed! Saved as 'router_history_weighted.index'


####running through the entire arch (adaptive rag)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import faiss
import pickle
from collections import Counter
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer

# =====================================================================
# 1. THE BOOT SEQUENCE (Loading all Models and Databases)
# =====================================================================
print("🚀 BOOTING ADAPTIVE-RAG PIPELINE...")

#Load LLM (Assuming Llama-3 8B)
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

# Load Embedder
print("Loading Semantic Embedder...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Load the Medical Knowledge Base (Database A)
print("Loading 250k Medical Knowledge Base...")
rag_index = faiss.read_index("pubmed_massive_faiss.index")
with open("pubmed_massive_mapping.pkl", "rb") as f:
    rag_mapping = pickle.load(f)

# Load the Router History (Database B)
print("Loading k-NN Router History...")
router_index = faiss.read_index("router_history_weighted.index")
with open("router_labels.pkl", "rb") as f:
    router_labels = pickle.load(f)

print("✅ ALL SYSTEMS GO.\n")

# =====================================================================
# 2. CORE FUNCTIONS
# =====================================================================

def extract_state_vector(question):
    """Generates draft, extracts entropy, and embeds text into a 386D vector."""
    # 1. Get Text Embedding (384D)
    semantic_vector = embedder.encode([question])[0]

    # 2. Get Uncertainty Metrics from LLM (2D)
    prompt = f"Question: {question}\nAnswer (Yes/No/Maybe):"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=10,
            return_dict_in_generate=True, output_scores=True,
            pad_token_id=tokenizer.eos_token_id
        )

    draft_answer = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True).split("Answer (Yes/No/Maybe):")[-1].strip()

    token_entropies = []
    for step_logits in outputs.scores:
        probs = F.softmax(step_logits[0], dim=-1)
        entropy = -torch.sum(probs * torch.log(probs + 1e-10))
        token_entropies.append(entropy.item())

    mean_ent = np.mean(token_entropies)
    var_ent = np.var(token_entropies)

    # 3. Fuse into 386D State Vector
    state_vector = np.concatenate((semantic_vector, [mean_ent * 50.0, var_ent * 50.0])).astype('float32')

    return draft_answer, state_vector, mean_ent, var_ent

def adaptive_router(state_vector, k=5):
    """Queries history to vote on hallucination risk."""
    query_vector = np.array([state_vector]).astype('float32')
    distances, indices = router_index.search(query_vector, k)

    neighbor_labels = router_labels[indices[0]]
    vote_counts = Counter(neighbor_labels)
    hallucination_votes = vote_counts.get(1, 0)

    # MEDICAL THRESHOLD: We are hyper-paranoid about hallucinations.
# If even 2 out of 5 historically similar queries failed, we hit the brakes!
    return "RAG" if hallucination_votes >= 1 else "PARAMETRIC", hallucination_votes

def execute_rag(question):
    """Searches medical DB and forces LLM to use context."""
    query_emb = np.array([embedder.encode(question)]).astype('float32')
    faiss.normalize_L2(query_emb)

    distances, indices = rag_index.search(query_emb, k=1) # Get top 1 abstract
    retrieved_idx = indices[0][0]
    retrieved_text = rag_mapping['texts'][retrieved_idx]

    # Force LLM to answer using only the retrieved text
    rag_prompt = f"Context: {retrieved_text}\nQuestion: {question}\nBased strictly on the context, Answer:"
    inputs = tokenizer(rag_prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)

    final_answer = tokenizer.decode(outputs[0], skip_special_tokens=True).split("Answer:")[-1].strip()
    return final_answer, retrieved_text

# =====================================================================
# 3. LIVE SYSTEM EXECUTION (Run this for your Midterm Screenshot!)
# =====================================================================

# Put a tricky medical question here to test the system
user_query = "Are the long-term results of the transanal pull-through equal to those of the transabdominal pull-through?"

print("="*70)
print(f"📥 INCOMING QUERY: {user_query}")
print("="*70)

# Step 1: Intercept and Extract
print("\n⚙️  Step 1: Extracting LLM Internal State...")
draft_ans, state_vec, m_ent, v_ent = extract_state_vector(user_query)
print(f"   Draft Parametric Answer: {draft_ans}")
print(f"   Mean Entropy: {m_ent:.4f} | Varentropy: {v_ent:.4f}")

# Step 2: Route via k-NN
print("\n🚦 Step 2: Routing via k-NN History...")
routing_decision, bad_votes = adaptive_router(state_vec)
print(f"   Precedent: {bad_votes} out of 5 similar past queries resulted in hallucinations.")
print(f"   Decision: >> TRIGGER {routing_decision} <<")

# Step 3: Final Output
print("\n📝 Step 3: Final Generation...")
if routing_decision == "RAG":
    final_answer, abstract_used = execute_rag(user_query)
    print(f"   [RAG Activated] Retrieved Abstract: {abstract_used[:100]}...")
    print(f"\n✅ FINAL GROUNDED ANSWER: {final_answer}")
else:
    print(f"\n✅ FINAL PARAMETRIC ANSWER: {draft_ans}")
print("="*70)

🚀 BOOTING ADAPTIVE-RAG PIPELINE...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading Semantic Embedder...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading 250k Medical Knowledge Base...
Loading k-NN Router History...
✅ ALL SYSTEMS GO.

📥 INCOMING QUERY: Are the long-term results of the transanal pull-through equal to those of the transabdominal pull-through?

⚙️  Step 1: Extracting LLM Internal State...
   Draft Parametric Answer: Yes
Explanation: The transanal pull-through (
   Mean Entropy: 0.3453 | Varentropy: 0.1488

🚦 Step 2: Routing via k-NN History...
   Precedent: 0 out of 5 similar past queries resulted in hallucinations.
   Decision: >> TRIGGER PARAMETRIC <<

📝 Step 3: Final Generation...

✅ FINAL PARAMETRIC ANSWER: Yes
Explanation: The transanal pull-through (


In [ ]:
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm

print("📊 INITIALIZING ADAPTIVE-RAG BENCHMARK...")

# 1. Load the evaluation dataset
dataset = load_dataset("pubmed_qa", "pqa_labeled", split="train")
df_eval = pd.DataFrame(dataset)

# Sample 100 random questions for a quick benchmark (change this to 500 for the final paper)
df_test = df_eval.sample(n=100, random_state=42).reset_index(drop=True)

# 2. Score trackers
metrics = {
    "PARAMETRIC": {"total": 0, "correct": 0},
    "RAG": {"total": 0, "correct": 0}
}

print(f"Executing Benchmark on {len(df_test)} queries...\n")

# 3. The Evaluation Loop
for idx, row in tqdm(df_test.iterrows(), total=len(df_test)):
    question = row['question']
    ground_truth = row['final_decision'].strip().lower() # 'yes', 'no', or 'maybe'

    # Step A: Intercept & Extract
    draft_ans, state_vec, _, _ = extract_state_vector(question)

    # Step B: Route
    routing_decision, _ = adaptive_router(state_vec)
    metrics[routing_decision]["total"] += 1

    # Step C: Final Execution & Verification
    final_answer = ""
    if routing_decision == "RAG":
        final_answer, _ = execute_rag(question)
    else:
        final_answer = draft_ans

    # Step D: Grade the Output
    # We check if the true 'yes/no' is inside the model's generated text
    if ground_truth in final_answer.lower():
        metrics[routing_decision]["correct"] += 1

# 4. Calculate Final Metrics
total_queries = len(df_test)
total_correct = metrics["PARAMETRIC"]["correct"] + metrics["RAG"]["correct"]
overall_accuracy = total_correct / total_queries

para_acc = metrics["PARAMETRIC"]["correct"] / max(1, metrics["PARAMETRIC"]["total"])
rag_acc = metrics["RAG"]["correct"] / max(1, metrics["RAG"]["total"])

print("\n" + "="*50)
print(" 🏆 ADAPTIVE-RAG BENCHMARK RESULTS")
print("="*50)
print(f"Overall System Accuracy: {overall_accuracy * 100:.1f}%\n")

print(f"🟢 PARAMETRIC PATH (LLM Memory):")
print(f"   Queries Routed: {metrics['PARAMETRIC']['total']} ({metrics['PARAMETRIC']['total']/total_queries*100:.1f}%)")
print(f"   Path Accuracy:  {para_acc * 100:.1f}%")

print(f"\n🚨 RAG PATH (Vector Database):")
print(f"   Queries Routed: {metrics['RAG']['total']} ({metrics['RAG']['total']/total_queries*100:.1f}%)")
print(f"   Path Accuracy:  {rag_acc * 100:.1f}%")
print("="*50)

📊 INITIALIZING ADAPTIVE-RAG BENCHMARK...
Executing Benchmark on 100 queries...



100%|██████████| 100/100 [00:39<00:00,  2.55it/s]


 🏆 ADAPTIVE-RAG BENCHMARK RESULTS
Overall System Accuracy: 69.0%

🟢 PARAMETRIC PATH (LLM Memory):
   Queries Routed: 100 (100.0%)
   Path Accuracy:  69.0%

🚨 RAG PATH (Vector Database):
   Queries Routed: 0 (0.0%)
   Path Accuracy:  0.0%
